In [6]:
from datasets import load_dataset
from itertools import islice

splits = get_dataset_split_names("eltorio/ROCOv2-radiology")
print(splits)

dataset = load_dataset(
    "eltorio/ROCOv2-radiology",
    streaming = True,
    split = "train"
)

dataset_shuffle = dataset.shuffle(seed = 42, buffer_size = 5000)
samples = list(islice(dataset_shuffle,1000))

dataset = load_dataset(
    "eltorio/ROCOv2-radiology",
    streaming = True,
    split = "validation"
)

dataset_shuffle = dataset.shuffle(seed = 42, buffer_size = 5000)
val_samples = list(islice(dataset_shuffle,200))

['train', 'validation', 'test']


In [7]:
train_ids = {sample["image_id"] for sample in samples}
val_ids = {sample["image_id"] for sample in val_samples}

overlap = train_ids & val_ids

print("Unique train IDs:", len(train_ids))
print("Unique validation IDs:", len(val_ids))
print("Overlapping IDs:", len(overlap))



Unique train IDs: 1000
Unique validation IDs: 200
Overlapping IDs: 0


In [8]:
import hashlib
import numpy as np

def image_hash(image):
    image = image.convert("RGB")
    array = np.asarray(image)
    return hashlib.sha256(array.tobytes() + str(array.shape).encode()).hexdigest()

train_hashes = {image_hash(s["image"]) for s in samples}
val_hashes = {image_hash(s["image"]) for s in val_samples}

print("Unique train image hashes:", len(train_hashes))
print("Unique validation image hashes:", len(val_hashes))
print("Cross-split duplicate hashes:", len(train_hashes & val_hashes))

Unique train image hashes: 1000
Unique validation image hashes: 200
Cross-split duplicate hashes: 0


In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
image = samples[0]["image"].convert("RGB")

pixel_values = transform(image)

print(pixel_values.shape)
pixel_values = pixel_values.unsqueeze(dim=0)

print(pixel_values.shape)

torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


In [11]:
from torch.utils.data import Dataset

class MedicalDataset(Dataset):
    def __init__(self,samples,transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return(len(self.samples))

    def __getitem__(self,index):
        sample = self.samples[index]
        return {
            "image": self.transform(sample["image"].convert("RGB")),
            "caption": sample["caption"],
            "image_id": sample["image_id"]
        }

In [12]:
dataset = MedicalDataset(samples, transform=transform)

In [13]:
def collate_fn(batch,tokenizer):
    images = [sample["image"] for sample in batch]
    captions = [sample["caption"] for sample in batch]
    image_ids = [sample["image_id"] for sample in batch]
    image_batch = torch.stack(images,dim = 0)
    encoded = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    return{ "pixel_values": image_batch,
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "image_ids": image_ids}

In [14]:
from torch.utils.data import DataLoader
from functools import partial
from transformers import AutoTokenizer
text_model_name = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)
tokenizer = AutoTokenizer.from_pretrained(text_model_name)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)


In [15]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "facebook/dinov2-small"
)

print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size)
print("Patch size:", config.patch_size)
print("Layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("Image size:", config.image_size)

Model type: dinov2
Hidden size: 384
Patch size: 14
Layers: 12
Attention heads: 6
Image size: 518


In [16]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("facebook/dinov2-small")

print(type(model))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

model.eval()

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 3194.31it/s]

<class 'transformers.models.dinov2.modeling_dinov2.Dinov2Model'>
Total parameters: 22056576


Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=True)
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Dinov2MLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=

In [17]:
from transformers import AutoConfig

text_model_name = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

text_config = AutoConfig.from_pretrained(text_model_name)

print("Model type:", text_config.model_type)
print("Hidden size:", text_config.hidden_size)
print("Layers:", text_config.num_hidden_layers)
print("Attention heads:", text_config.num_attention_heads)
print("Vocabulary size:", text_config.vocab_size)
print("Max positions:", text_config.max_position_embeddings)

Model type: bert
Hidden size: 768
Layers: 12
Attention heads: 12
Vocabulary size: 30522
Max positions: 512


In [18]:
from transformers import AutoModel

text_model = AutoModel.from_pretrained(text_model_name)
text_model.eval()

print("Total parameters:", sum(p.numel() for p in text_model.parameters()))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16628.15it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters: 109482240


In [45]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

image_projection = nn.Linear(384, 256)
text_projection = nn.Linear(768, 256)

optimizer = optim.AdamW(
    list(image_projection.parameters())
    + list(text_projection.parameters()),
    lr=1e-4
)

temperature = 0.07

for epoch in range(10):

    for step, batch in enumerate(loader):

        images = batch["pixel_values"]
        input_ids = batch["input_ids"]

        assert images.shape[1:] == (3, 224, 224)
        assert images.shape[0] == input_ids.shape[0]
        assert input_ids.shape[1] <= 128

        with torch.no_grad():
            outputs = model(pixel_values=batch["pixel_values"])
        hidden = outputs.last_hidden_state

        cls_embedding = hidden[:, 0, :]

        with torch.no_grad():
            text_outputs = text_model(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"]
                            )
        hidden = text_outputs.last_hidden_state

        text_cls = hidden[:, 0, :]
        mask = batch["attention_mask"]
        mask_expanded = mask.unsqueeze(-1)
        masked_hidden = hidden * mask_expanded
        sum_hidden = masked_hidden.sum(dim=1)
        token_counts = mask_expanded.sum(dim=1)
        text_mean = sum_hidden / token_counts.clamp(min=1)

        image_features = cls_embedding
        text_features = text_mean

        image_projected = image_projection(image_features)
        text_projected = text_projection(text_features)

        image_norm = F.normalize(image_projected, p=2, dim=-1)
        text_norm = F.normalize(text_projected, p=2, dim=-1)

        similarity = image_norm @ text_norm.T
        logits = similarity / temperature

        labels = torch.arange(images.shape[0])
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        optimizer.zero_grad()
        loss.backward()
        
        print(
        f"Step {step+1}: "
        f"Loss={loss.item():.4f}, "
        f"Image grad={image_projection.weight.grad.norm().item():.4f}, "
        f"Text grad={text_projection.weight.grad.norm().item():.4f}"
    )
        optimizer.step()

    #    if step == 2500:
    #        break


Step 1: Loss=1.6514, Image grad=7.8307, Text grad=11.2821
Step 2: Loss=1.4605, Image grad=3.2128, Text grad=4.2987
Step 3: Loss=1.5266, Image grad=2.8632, Text grad=3.7626
Step 4: Loss=1.3947, Image grad=1.7524, Text grad=2.5307
Step 5: Loss=1.5138, Image grad=3.5083, Text grad=4.4225
Step 6: Loss=1.4998, Image grad=2.1899, Text grad=2.9805
Step 7: Loss=1.4238, Image grad=2.1331, Text grad=2.9850
Step 8: Loss=1.5306, Image grad=5.6474, Text grad=7.8354
Step 9: Loss=1.4316, Image grad=2.6790, Text grad=3.7963
Step 10: Loss=1.4045, Image grad=3.7264, Text grad=5.0076
Step 11: Loss=1.3866, Image grad=2.1637, Text grad=2.8533
Step 12: Loss=1.4100, Image grad=3.2938, Text grad=4.3507
Step 13: Loss=1.3940, Image grad=2.5276, Text grad=3.2919
Step 14: Loss=1.4341, Image grad=2.9545, Text grad=3.5417
Step 15: Loss=1.3720, Image grad=2.4076, Text grad=3.3256
Step 16: Loss=1.3784, Image grad=2.0795, Text grad=2.8669
Step 17: Loss=1.3215, Image grad=2.2440, Text grad=3.0742
Step 18: Loss=1.4587, 

In [49]:
import torch
from pathlib import Path

Path("checkpoints").mkdir(exist_ok=True)

torch.save({
    "image_projection": image_projection.state_dict(),
    "text_projection": text_projection.state_dict(),
    "optimizer": optimizer.state_dict(),
    "temperature": temperature,
}, "checkpoints/day7_projection.pt")

In [32]:
val_dataset = MedicalDataset(val_samples, transform=transform)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)

print("Validation samples:", len(val_dataset))
print("Validation batches:", len(val_loader))

def recall_at_k(labels,similarity,k):
    topk = similarity.topk(k=k,dim = 1).indices
    correct = labels.unsqueeze(1) == topk
    hits = correct.any(dim=1)
    return hits.float().mean().item()



Validation samples: 200
Validation batches: 50


In [46]:
model.eval()
text_model.eval()
image_projection.eval()
text_projection.eval()

all_image_embeddings = []
all_text_embeddings = []

In [47]:
with torch.no_grad():

    for batch in val_loader:

        image_outputs = model(
            pixel_values=batch["pixel_values"]
        )

        text_outputs = text_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        # 1. Image CLS
        image_features = image_outputs.last_hidden_state[:, 0, :]

        # 2. Text masked mean pooling
        hidden = text_outputs.last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1)

        text_features = (
            (hidden * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

        # 3. Projection
        image_emb = image_projection(image_features)
        text_emb = text_projection(text_features)

        # 4. L2 normalization
        image_emb = F.normalize(image_emb, p=2, dim=-1)
        text_emb = F.normalize(text_emb, p=2, dim=-1)

        # 5. 保存当前Batch的Embedding
        all_image_embeddings.append(image_emb)
        all_text_embeddings.append(text_emb)

In [48]:
image_embeddings = torch.cat(all_image_embeddings, dim=0)
text_embeddings = torch.cat(all_text_embeddings, dim=0)
similarity = image_embeddings @ text_embeddings.T
print("Similarity:", similarity.shape)
labels = torch.arange(similarity.shape[0])
for k in [1, 5, 10]:
    i2t = recall_at_k(labels, similarity, k)
    t2i = recall_at_k(labels, similarity.T, k)

    print(f"R@{k}: I2T={i2t:.4f}, T2I={t2i:.4f}")

Similarity: torch.Size([200, 200])
R@1: I2T=0.0750, T2I=0.0500
R@5: I2T=0.3300, T2I=0.2850
R@10: I2T=0.4750, T2I=0.4900
